# Feature Selection & Cell Type Classification
This notebook demonstrates how to:
1. Load a single-cell dataset
2. Split into train/test sets
3. Select top-K features (genes) using ANOVA F-scores on the **training set only**
4. Apply the same feature selection to the test set
5. Train and evaluate a classifier

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data
Replace this cell with your own data loading. We expect:
- `X`: a (n_cells, n_genes) matrix
- `y`: a (n_cells,) array of cell-type labels

In [ ]:
# ── Option A: Load from an .h5ad file ──
# adata = sc.read_h5ad('your_dataset.h5ad')

# ── Option B: Use a built-in scanpy dataset for demo ──
adata = sc.datasets.pbmc3k_processed()

print(f"Dataset shape: {adata.shape}")
print(f"Cell types: {adata.obs['louvain'].nunique()} unique")
print(adata.obs['louvain'].value_counts())

In [ ]:
# ── Prepare X and y ──
# Adjust the obs column name to match your cell-type annotation
CELL_TYPE_COL = 'louvain'  # <-- change this to your column name

# Convert sparse matrix to dense if needed
from scipy.sparse import issparse
X = adata.X.toarray() if issparse(adata.X) else np.array(adata.X)

# Encode labels
le = LabelEncoder()
y = le.fit_transform(adata.obs[CELL_TYPE_COL])
label_names = le.classes_

print(f"X shape: {X.shape}")
print(f"Labels: {label_names}")

## 2. Train / Test Split
We use stratified splitting to maintain cell-type proportions in both sets.

In [ ]:
TEST_SIZE = 0.2
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,  # preserve class proportions
)

print(f"Train: {X_train.shape[0]} cells")
print(f"Test:  {X_test.shape[0]} cells")

## 3. Feature Selection (fit on train, transform both)

**Critical:** `SelectKBest` is fitted on the training data only to avoid data leakage. The same transformation is then applied to the test set.

In [ ]:
K_FEATURES = 128  # number of top genes to select

selector = SelectKBest(score_func=f_classif, k=K_FEATURES)

# Fit on TRAIN only
X_train_selected = selector.fit_transform(X_train, y_train)

# Apply the SAME selection to TEST
X_test_selected = selector.transform(X_test)

print(f"Train selected: {X_train_selected.shape}")
print(f"Test selected:  {X_test_selected.shape}")

In [ ]:
# ── Inspect selected genes ──
selected_indices = selector.get_support(indices=True)
selected_genes = adata.var_names[selected_indices]

# Show top 20 by F-score
scores = selector.scores_[selected_indices]
gene_scores = pd.DataFrame({'gene': selected_genes, 'F_score': scores})
gene_scores = gene_scores.sort_values('F_score', ascending=False).reset_index(drop=True)

print(f"\nTop 20 genes by F-score:")
gene_scores.head(20)

In [ ]:
# ── Visualize F-score distribution ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All gene scores
axes[0].hist(selector.scores_, bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('F-score Distribution (All Genes)')
axes[0].set_xlabel('F-score')
axes[0].set_ylabel('Count')
axes[0].axvline(np.sort(selector.scores_)[-K_FEATURES], color='red',
                linestyle='--', label=f'Top {K_FEATURES} threshold')
axes[0].legend()

# Selected gene scores
axes[1].barh(gene_scores['gene'][:20], gene_scores['F_score'][:20], color='coral')
axes[1].set_title(f'Top 20 Selected Genes')
axes[1].set_xlabel('F-score')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 4. Classification
We test three classifiers on the selected features.

In [ ]:
# ── Define classifiers ──
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=2000, C=1.0, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    'Linear SVM': LinearSVC(max_iter=5000, C=1.0, random_state=RANDOM_STATE),
}

results = {}

for name, clf in classifiers.items():
    # Scale features (important for LR and SVM)
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train_selected)
    X_te = scaler.transform(X_test_selected)

    clf.fit(X_tr, y_train)
    y_pred = clf.predict(X_te)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    results[name] = {'accuracy': acc, 'f1_weighted': f1, 'y_pred': y_pred}

    print(f"\n{'='*60}")
    print(f"{name}  |  Accuracy: {acc:.4f}  |  F1 (weighted): {f1:.4f}")
    print(f"{'='*60}")
    print(classification_report(y_test, y_pred, target_names=label_names))

In [ ]:
# ── Summary table ──
summary = pd.DataFrame({name: {k: v for k, v in vals.items() if k != 'y_pred'}
                         for name, vals in results.items()}).T
summary = summary.sort_values('f1_weighted', ascending=False)
print("\nModel Comparison:")
summary.style.format('{:.4f}').highlight_max(axis=0, color='lightgreen')

In [ ]:
# ── Confusion matrix for the best model ──
best_model_name = summary.index[0]
best_preds = results[best_model_name]['y_pred']

fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(y_test, best_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=label_names)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
ax.set_title(f'Confusion Matrix — {best_model_name}')
plt.tight_layout()
plt.show()

## 5. (Optional) Sweep over different K values
See how the number of selected features affects performance.

In [ ]:
k_values = [16, 32, 64, 128, 256, 512]
sweep_results = []

for k in k_values:
    sel = SelectKBest(f_classif, k=min(k, X_train.shape[1]))
    X_tr_k = sel.fit_transform(X_train, y_train)
    X_te_k = sel.transform(X_test)

    scaler = StandardScaler()
    X_tr_k = scaler.fit_transform(X_tr_k)
    X_te_k = scaler.transform(X_te_k)

    clf = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    clf.fit(X_tr_k, y_train)
    y_pred = clf.predict(X_te_k)

    sweep_results.append({
        'k': k,
        'accuracy': accuracy_score(y_test, y_pred),
        'f1_weighted': f1_score(y_test, y_pred, average='weighted'),
    })

sweep_df = pd.DataFrame(sweep_results)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sweep_df['k'], sweep_df['accuracy'], 'o-', label='Accuracy')
ax.plot(sweep_df['k'], sweep_df['f1_weighted'], 's--', label='F1 (weighted)')
ax.set_xlabel('Number of Selected Features (k)')
ax.set_ylabel('Score')
ax.set_title('Feature Selection Sweep — Logistic Regression')
ax.legend()
ax.set_xscale('log', base=2)
ax.set_xticks(k_values)
ax.set_xticklabels(k_values)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(sweep_df.to_string(index=False))

## 6. Using a Pipeline (Recommended for Production)
Wrapping everything in a scikit-learn `Pipeline` ensures no data leakage and makes it easy to deploy.

In [ ]:
pipe = Pipeline([
    ('feature_selection', SelectKBest(f_classif, k=128)),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

pipe.fit(X_train, y_train)
y_pred_pipe = pipe.predict(X_test)

print(f"Pipeline Accuracy: {accuracy_score(y_test, y_pred_pipe):.4f}")
print(f"Pipeline F1:       {f1_score(y_test, y_pred_pipe, average='weighted'):.4f}")

In [ ]:
# ── Save the pipeline for later use ──
import joblib
joblib.dump(pipe, 'cell_type_classifier_pipeline.pkl')
joblib.dump(le, 'label_encoder.pkl')
print("Pipeline and label encoder saved.")

## 7. Save 128-dim Features as {label: vector} Dictionary
Export all cells' selected features grouped by their cell-type label.

In [ ]:
import pickle

# ── Apply the trained selector to ALL data ──
# (selector was fit on train set — no leakage)
X_all_selected = selector.transform(X)  # (n_cells, 128)

# ── Build dictionary: {label_string: array of shape (n_cells_with_that_label, 128)} ──
label_to_vectors = {}
for label_idx, label_name in enumerate(label_names):
    mask = y == label_idx
    label_to_vectors[label_name] = X_all_selected[mask]  # (n_cells_of_type, 128)

# ── Preview ──
for label, vecs in label_to_vectors.items():
    print(f"{label:25s} → {vecs.shape[0]:5d} cells × {vecs.shape[1]} features")

# ── Save ──
output_path = 'cell_type_features_128d.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(label_to_vectors, f)

print(f'\nSaved to {output_path}')

In [ ]:
# ── Verify: reload and inspect ──
with open(output_path, 'rb') as f:
    loaded = pickle.load(f)

for label, vecs in loaded.items():
    print(f"{label:25s} → shape {vecs.shape}, dtype {vecs.dtype}")